In [ ]:
!pip install vaderSentiment gensim pyLDAvis -q

import requests, time, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import requests, time, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import os, json, pandas as pd
import re



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 38.8 MB/s eta 0:00:00


In [ ]:


API_KEY = "zhiSHk1f6u0pnTrMlq35w4OQhFRWmKTYT0ozArMjD1KaYA66kCbx6euOcrzFXatR3ZArLpwlEcINW9XJKV6uPiKGn5nMcw7RAMO561c1wboboI2MPoF38k8iIHMihc6l"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}
BASE_URL = "https://api.zembra.io"

In [ ]:
bay_area_locations = [
    "San Francisco, CA", "Oakland, CA", "San Jose, CA",
    "Berkeley, CA", "Fremont, CA", "Hayward, CA",
    "Santa Clara, CA", "Sunnyvale, CA", "Daly City, CA"
]

all_shops = {}

for location in bay_area_locations:
    r = requests.get(f"{BASE_URL}/listing/find", headers=HEADERS, params={
        "query": "auto repair shop",
        "location": location,
        "networks[]": "yelp"
    })
    shops = r.json().get("data", {}).get("yelp", [])
    for shop in shops:
        slug = shop.get("slug") or shop.get("rawData", {}).get("alias")
        name = shop.get("name")
        if slug and slug not in all_shops:
            all_shops[slug] = name
    print(f"{location}: {len(shops)} shops found")
    time.sleep(0.5)

print(f"\nTotal unique shops: {len(all_shops)}")

San Francisco, CA: 10 shops found
Oakland, CA: 10 shops found
San Jose, CA: 10 shops found
Berkeley, CA: 10 shops found
Fremont, CA: 10 shops found
Hayward, CA: 10 shops found
Santa Clara, CA: 10 shops found
Sunnyvale, CA: 10 shops found
Daly City, CA: 10 shops found

Total unique shops: 83


In [ ]:
jobs = []
for slug, name in all_shops.items():
    r = requests.post(f"{BASE_URL}/reviews", headers=HEADERS, json={
        "network": "yelp",
        "slug": slug
    })
    result = r.json()
    job_id = result.get("data", {}).get("job", {}).get("jobId")
    jobs.append({"name": name, "slug": slug, "jobId": job_id})
    print(f"✓ {name} → {job_id}")
    time.sleep(0.5)

print(f"\nTotal jobs created: {len(jobs)}")

✓ Sunset Auto Care → 9681dcb8-8efd-46f7-97b5-368b5d9fb668
✓ Stress-Free Auto Care → 5ebc7c0d-0582-4bda-82eb-c9f77615178d
✓ Atech Auto Repair → fc1e7f54-ab61-41ae-b49a-9670d2c6d628
✓ KSH Automotive → 2ced6c45-fb33-424d-b7b6-0b49c44d4214
✓ Metric Motors → 14420a26-61fe-482a-b5e3-42166ea2bf3a
✓ K & C Auto Service → c3ba5a19-26fa-420f-bc0c-dd4d311018f2
✓ Hayes Auto Repair → 21f835e1-ced0-4cde-a3a7-19acc7f87393
✓ Sunset 76 Auto Repair & Tire Center → 83b189d1-f1ac-4499-8752-2625a0d1d8cb
✓ Eddy's Auto Services → 3fed7a2d-3935-4ba8-a16c-582e5e6da461
✓ Masonic Smog and Repair → ba81adfd-d3cb-49c1-a66a-46d84db19fb3
✓ First Choice Auto Repair & Brakes → 13bf3bdc-3050-434f-ab1f-75f32a86dcc7
✓ TMS Automotive → d71bbb4e-5950-46a1-95ab-94e2f0209a1d
✓ Glenview Automotive → cec43fb8-32aa-4e28-a764-aabac95f199d
✓ 1701 Auto Care → adbae65a-bfdd-488e-bb98-d00fe09b5298
✓ Car Care Service → 4c629ea9-0c8e-464b-bc05-7df7720d5f40
✓ Nayarit Transmission & Auto Repair → a7346588-f6ba-419b-bab0-605a0b8fc03c
✓ Ye

In [ ]:
# Filter only successful jobs
successful_jobs = [j for j in jobs if j["jobId"] is not None]
print(f"Successful jobs: {len(successful_jobs)}")
for j in successful_jobs:
    print(f"  {j['name']} → {j['jobId']}")

Successful jobs: 33
  Sunset Auto Care → 9681dcb8-8efd-46f7-97b5-368b5d9fb668
  Stress-Free Auto Care → 5ebc7c0d-0582-4bda-82eb-c9f77615178d
  Atech Auto Repair → fc1e7f54-ab61-41ae-b49a-9670d2c6d628
  KSH Automotive → 2ced6c45-fb33-424d-b7b6-0b49c44d4214
  Metric Motors → 14420a26-61fe-482a-b5e3-42166ea2bf3a
  K & C Auto Service → c3ba5a19-26fa-420f-bc0c-dd4d311018f2
  Hayes Auto Repair → 21f835e1-ced0-4cde-a3a7-19acc7f87393
  Sunset 76 Auto Repair & Tire Center → 83b189d1-f1ac-4499-8752-2625a0d1d8cb
  Eddy's Auto Services → 3fed7a2d-3935-4ba8-a16c-582e5e6da461
  Masonic Smog and Repair → ba81adfd-d3cb-49c1-a66a-46d84db19fb3
  First Choice Auto Repair & Brakes → 13bf3bdc-3050-434f-ab1f-75f32a86dcc7
  TMS Automotive → d71bbb4e-5950-46a1-95ab-94e2f0209a1d
  Glenview Automotive → cec43fb8-32aa-4e28-a764-aabac95f199d
  1701 Auto Care → adbae65a-bfdd-488e-bb98-d00fe09b5298
  Car Care Service → 4c629ea9-0c8e-464b-bc05-7df7720d5f40
  Nayarit Transmission & Auto Repair → a7346588-f6ba-419b-ba